# EduPro — 02. Learner Feature Engineering & Segmentation

This notebook uses the same reusable data pipeline as the Streamlit application. It builds learner-level profiles, inspects engineered features, evaluates candidate K-Means cluster counts, compares Ward hierarchical clustering, and visualizes clusters with PCA.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

ROOT = Path.cwd()
if not (ROOT / "dashboard").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from dashboard.data_loader import build_data, save_processed_data

D = build_data()
users, courses, transactions = D["users"], D["courses"], D["transactions"]
learners = D["learners"].copy()
enriched = D["enriched"].copy()
print("Learner profiles:", learners.shape)
display(learners.head())

In [ ]:
feature_columns = [
    "Age", "total_enrollments", "unique_courses", "total_spend",
    "avg_spend_per_enrollment", "avg_course_rating", "category_diversity",
    "average_courses_per_category", "enrollment_frequency", "learning_depth_index"
]
display(learners[feature_columns].describe().T)

In [ ]:
# Inspect engineered learner behaviour features
feature_summary = learners.groupby("segment_name").agg(
    learners=("UserID", "nunique"),
    mean_enrollments=("total_enrollments", "mean"),
    mean_spend=("total_spend", "mean"),
    mean_category_diversity=("category_diversity", "mean"),
    mean_learning_depth=("learning_depth_index", "mean"),
    mean_rating=("avg_course_rating", "mean")
).round(3)
display(feature_summary)

In [ ]:
# Prepare numeric matrix and standardize features
X = learners[feature_columns].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True)).fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaled feature matrix:", X_scaled.shape)

In [ ]:
# Candidate K evaluation using silhouette score
candidate_scores = []
max_k = min(8, len(learners)-1)
for k in range(2, max_k+1):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    if len(set(labels)) > 1:
        candidate_scores.append({"k":k, "silhouette":silhouette_score(X_scaled, labels)})

scores_df = pd.DataFrame(candidate_scores)
display(scores_df)
if not scores_df.empty:
    scores_df.plot(x="k", y="silhouette", marker="o", legend=False, figsize=(7,4),
                   title="Silhouette score by number of clusters")
    plt.ylabel("Silhouette score")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough distinct learner profiles to evaluate multiple clusters.")

In [ ]:
# Fit selected K-Means and compare with Ward hierarchical clustering
if not scores_df.empty:
    best_k = int(scores_df.loc[scores_df["silhouette"].idxmax(), "k"])
    km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    kmeans_labels = km.fit_predict(X_scaled)
    ward_labels = AgglomerativeClustering(n_clusters=best_k, linkage="ward").fit_predict(X_scaled)
    print("Selected K:", best_k)
    print("K-Means silhouette:", round(silhouette_score(X_scaled, kmeans_labels), 4))
    print("Ward silhouette:", round(silhouette_score(X_scaled, ward_labels), 4))
else:
    best_k, kmeans_labels = 1, np.zeros(len(learners), dtype=int)
    ward_labels = kmeans_labels
    print("Single-segment fallback used for this small/low-variation dataset.")

learners["notebook_segment_id"] = kmeans_labels
display(learners.groupby("notebook_segment_id")[feature_columns].mean().round(2))

In [ ]:
# PCA is used only to visualize the standardized feature space
if X_scaled.shape[0] >= 2 and X_scaled.shape[1] >= 2:
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X_scaled)
    learners["pca_1"], learners["pca_2"] = coords[:,0], coords[:,1]
    print("Explained variance by first two components:", pca.explained_variance_ratio_,
          "Total:", round(float(pca.explained_variance_ratio_.sum()), 4))
    for sid, group in learners.groupby("notebook_segment_id"):
        plt.scatter(group["pca_1"], group["pca_2"], label=f"Segment {sid}", alpha=.7)
    plt.title("EduPro learner segments — PCA projection")
    plt.xlabel("PCA component 1")
    plt.ylabel("PCA component 2")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Save the cleaned and engineered datasets for reuse by the dashboard and later analysis
processed_dir = ROOT / "Data" / "Processed"
saved_files = save_processed_data(D, processed_dir)

print("Processed datasets saved:")
for file_path in saved_files:
    print("-", file_path.relative_to(ROOT))


## Interpretation checklist

Give each cluster a descriptive persona only after inspecting its mean/median features. Possible interpretations include exploratory, specialized, advanced/deep, or multi-category learners. These are interpretations, not target labels.

Record the selected K, silhouette values, cluster sizes, feature profiles, and limitations for the internship research paper.